In [109]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import mean_absolute_error, mean_squared_error, classification_report

In [110]:
data_path = "../data/processed/data_processed.csv"
df = pd.read_csv(data_path, parse_dates=['date'])

In [111]:
df_sort = df.sort_values(by=['date', 'city']).reset_index(drop=True)
df_sort.head(24)

,city,latitude,longitude,date,temperature_2m_mean,rain_sum,precipitation_hours,weather_code,wind_speed_10m_mean,relative_humidity_2m_mean,...,dewpoint_2m_mean_lag_2,surface_pressure_mean_lag_2,cloudcover_mean_lag_2,wind_speed_10m_mean_lag_2,temperature_2m_mean_rolling_mean_3,relative_humidity_2m_mean_rolling_mean_3,dewpoint_2m_mean_rolling_mean_3,surface_pressure_mean_rolling_mean_3,cloudcover_mean_rolling_mean_3,wind_speed_10m_mean_rolling_mean_3
0,0,10.5417,107.2429,2014-01-08,-0.351351,0.125,1.0,1,0.510638,-0.773585,...,-0.993846,-0.197183,-0.412916,1.042553,-0.441441,-0.918239,-0.943932,-0.144422,-0.002712,6.950355e-01
1,1,16.0544,108.2022,2014-01-08,-1.486486,0.000,0.0,0,-0.212766,0.509434,...,-1.526154,0.820926,-1.465548,0.425532,-1.549550,0.246541,-1.275214,0.840599,-1.167218,2.198582e-01
2,2,10.9465,106.8340,2014-01-08,-0.162162,0.000,0.0,0,-0.617021,-1.007547,...,-0.899487,0.256204,-0.608187,-0.361702,-0.189189,-1.044182,-0.884103,0.316790,-0.261378,-3.971631e-01
3,3,21.0278,105.8342,2014-01-08,-2.324324,0.750,5.0,1,0.340426,-0.332075,...,-1.757949,0.393696,-0.325451,0.659574,-1.978604,-0.030189,-1.811282,0.622625,-0.100347,2.553191e-01
4,4,10.7769,106.7009,2014-01-08,-0.081081,0.000,0.0,0,-0.489362,-1.132075,...,-0.925128,0.156271,-0.533944,-0.276596,-0.162162,-1.073270,-0.923761,0.213280,-0.178998,-2.588652e-01
5,5,16.4637,107.5909,2014-01-08,-1.378378,0.000,0.0,0,-0.531915,0.596226,...,-1.417436,0.684105,-1.774727,0.659574,-1.342342,0.167296,-1.130598,0.736195,-1.343165,7.092199e-02
6,6,12.2585,109.0526,2014-01-08,-1.027027,0.000,0.0,0,1.212766,-0.033962,...,-1.368205,0.594232,-1.160437,1.255319,-1.027027,-0.402516,-1.155556,0.596691,-0.720061,1.127660e+00
7,7,10.5336,106.4110,2014-01-08,-0.243243,0.000,0.0,0,-0.489362,-0.728302,...,-0.779487,0.305835,-0.364099,-0.255319,-0.288288,-0.838994,-0.789060,0.358596,-0.087465,-2.836879e-01
8,8,15.5394,108.0191,2014-01-08,-1.432432,0.000,0.0,0,-0.702128,0.384906,...,-1.510769,0.179745,-1.433003,-0.297872,-1.423423,0.084277,-1.268718,0.197407,-0.971608,-4.680851e-01
9,9,21.0064,107.2925,2014-01-08,-2.540541,0.700,2.0,1,1.664894,-0.177358,...,-1.758974,0.799463,-0.101704,0.297872,-2.333333,0.394969,-1.891966,0.963783,-0.046784,6.187943e-01


In [112]:
def split_data(df, val_year = 1):
    years = df['date'].dt.year.unique()
    val_years = years[-val_year:]
    train_data = df[~df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    val_data = df[df['date'].dt.year.isin(val_years)].reset_index(drop=True)
    return train_data, val_data

In [113]:
train_data, val_data = split_data(df_sort, val_year=1)

In [114]:
train_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 43740 entries, 0 to 43739
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      43740 non-null  int64         
 1   latitude                                  43740 non-null  float64       
 2   longitude                                 43740 non-null  float64       
 3   date                                      43740 non-null  datetime64[ns]
 4   temperature_2m_mean                       43740 non-null  float64       
 5   rain_sum                                  43740 non-null  float64       
 6   precipitation_hours                       43740 non-null  float64       
 7   weather_code                              43740 non-null  int64         
 8   wind_speed_10m_mean                       43740 non-null  float64       
 9   relative_humidity_2m_mean   

In [115]:
val_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4392 entries, 0 to 4391
Data columns (total 58 columns):
 #   Column                                    Non-Null Count  Dtype         
---  ------                                    --------------  -----         
 0   city                                      4392 non-null   int64         
 1   latitude                                  4392 non-null   float64       
 2   longitude                                 4392 non-null   float64       
 3   date                                      4392 non-null   datetime64[ns]
 4   temperature_2m_mean                       4392 non-null   float64       
 5   rain_sum                                  4392 non-null   float64       
 6   precipitation_hours                       4392 non-null   float64       
 7   weather_code                              4392 non-null   int64         
 8   wind_speed_10m_mean                       4392 non-null   float64       
 9   relative_humidity_2m_mean     

### CONFIG

In [116]:
# Col to predict
target_column = 'rain_sum'
side_target = ['precipitation_hours', 'weather_code', "date"]

# Define training and testing sets
X_train = train_data.drop(columns=side_target + [target_column])
y_rain = train_data[target_column].values
y_precip_hours = train_data['precipitation_hours'].values
y_weather_code = train_data['weather_code'].values
X_test = val_data.drop(columns=side_target + [target_column])
y_test_rain = val_data[target_column].values
y_test_precip_hours = val_data['precipitation_hours'].values
y_test_weather_code = val_data['weather_code'].values


# METRICS


# XGBOOST

1. Model

In [117]:
import xgboost as xgb

model_rain = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='reg:squarederror',
    enable_categorical=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

model_precip_hours = xgb.XGBRegressor(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='reg:squarederror',
    enable_categorical=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

model_weather_code = xgb.XGBClassifier(
    n_estimators=1000,
    learning_rate=0.05,
    max_depth=6,
    objective='multi:softmax',
    num_class=len(train_data['weather_code'].unique()),
    enable_categorical=True,
    device='cuda' if torch.cuda.is_available() else 'cpu'
)

2. Training

In [118]:
model_rain.fit(X_train, y_rain)
model_precip_hours.fit(X_train, y_precip_hours)
model_weather_code.fit(X_train, y_weather_code)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device='cuda', early_stopping_rounds=None,
              enable_categorical=True, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.05, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=6,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=1000,
              n_jobs=None, num_class=2, ...)

3. Evaluation

3.1 Raining result

In [119]:
y_pred_rain = model_rain.predict(X_test)
mae = mean_absolute_error(y_true=y_test_rain, y_pred=y_pred_rain)
mse = mean_squared_error(y_test_rain, y_pred_rain)
print(f"Rain Sum - MAE: {mae}, MSE: {mse}")

Rain Sum - MAE: 0.46033349443220273, MSE: 6.620329936049829


In [120]:
cols = X_train.columns.tolist()
for feat, imp in zip(cols, model_rain.feature_importances_):
    print(f"{feat}: {imp}")

city: 0.0010767844505608082
latitude: 0.0025165085680782795
longitude: 0.0008312833379022777
temperature_2m_mean: 0.010642483830451965
wind_speed_10m_mean: 0.029419459402561188
relative_humidity_2m_mean: 0.06445301324129105
dewpoint_2m_mean: 0.003099778899922967
surface_pressure_mean: 0.0021879991982132196
cloudcover_mean: 0.09189289808273315
year: 0.006628607399761677
month_sin: 0.001179325394332409
month_cos: 0.0019927220419049263
day_sin: 0.009342703968286514
day_cos: 0.0033633492421358824
dayofweek_sin: 0.004647884983569384
dayofweek_cos: 0.0029740461613982916
quarter_sin: 0.0002261106128571555
quarter_cos: 0.0005969181656837463
rain_sum_lag_1: 0.07040264457464218
weather_code_lag_1: 0.0113489655777812
precipitation_hours_lag_1: 0.0023242426104843616
rain_sum_lag_2: 0.08876098692417145
weather_code_lag_2: 0.00034123449586331844
precipitation_hours_lag_2: 0.0041169337928295135
rain_sum_lag_3: 0.009299028664827347
weather_code_lag_3: 0.00035591726191341877
precipitation_hours_lag_3: 

3.2 precipitation_hours result

In [121]:
y_pred_precip_hours = model_precip_hours.predict(X_test)
mae = mean_absolute_error(y_true=y_test_precip_hours, y_pred=y_pred_rain)
mse = mean_squared_error(y_test_precip_hours, y_pred_rain)
print(f"Precipitation Hours - MAE: {mae}, MSE: {mse}")

Precipitation Hours - MAE: 3.69480621347879, MSE: 43.605301141516584


3.3 Weather Code result

In [122]:
y_pred_weather_code = model_weather_code.predict(X_test)
rp = classification_report(y_test_weather_code, y_pred_weather_code)

In [123]:
print(rp)

              precision    recall  f1-score   support

           0       1.00      1.00      1.00      1151
           1       1.00      1.00      1.00      3241

    accuracy                           1.00      4392
   macro avg       1.00      1.00      1.00      4392
weighted avg       1.00      1.00      1.00      4392



4. Save Model

In [124]:
import joblib
joblib.dump(model_rain, '../models/xgboost_rain.pkl')
joblib.dump(model_precip_hours, '../models/xgboost_precip_hours.pkl')
joblib.dump(model_weather_code, '../models/xgboost_weather_code.pkl')

['../models/xgboost_weather_code.pkl']

# RANDOM_FOREST

In [125]:
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier

model_rain = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf = 10,
    min_samples_split = 20,
    max_features = 'sqrt',
    n_jobs=-1)

model_precip_hours = RandomForestRegressor(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf = 10,
    min_samples_split = 20,
    max_features = 'sqrt',
    n_jobs=-1)

model_weather_code = RandomForestClassifier(
    n_estimators=500,
    max_depth=6,
    min_samples_leaf = 10,
    min_samples_split = 20,
    max_features = 'sqrt',
    class_weight='balanced',
    n_jobs=-1)


In [126]:
model_rain.fit(X_train, y_rain)
model_precip_hours.fit(X_train, y_precip_hours)
model_weather_code.fit(X_train, y_weather_code)

RandomForestClassifier(class_weight='balanced', max_depth=6,
                       min_samples_leaf=10, min_samples_split=20,
                       n_estimators=500, n_jobs=-1)

In [127]:
y_pred_rain = model_rain.predict(X_test)
mae = mean_absolute_error(y_true=y_test_rain, y_pred=y_pred_rain)
mse = mean_squared_error(y_test_rain, y_pred_rain)
print(f"Rain Sum - MAE: {mae}, MSE: {mse}")

Rain Sum - MAE: 2.99637303057131, MSE: 36.44176826248233


In [128]:
y_pred_precip_hours = model_precip_hours.predict(X_test)
mae = mean_absolute_error(y_true=y_test_precip_hours, y_pred=y_pred_rain)
mse = mean_squared_error(y_test_precip_hours, y_pred_rain)
print(f"Precipitation Hours - MAE: {mae}, MSE: {mse}")

Precipitation Hours - MAE: 2.85700030016085, MSE: 18.991095839685872


In [129]:
y_pred_weather_code = model_weather_code.predict(X_test)
rp = classification_report(y_test_weather_code, y_pred_weather_code)
print(rp)

              precision    recall  f1-score   support

           0       0.80      1.00      0.88      1151
           1       1.00      0.91      0.95      3241

    accuracy                           0.93      4392
   macro avg       0.90      0.95      0.92      4392
weighted avg       0.95      0.93      0.93      4392



# ROLLING PREDICT

In [130]:
target_cols = ["rain_sum", "weather_code", "precipitation_hours"]
dynamic_cols = ['temperature_2m_mean', 
    'relative_humidity_2m_mean', 
    'dewpoint_2m_mean', 
    'surface_pressure_mean', 
    'cloudcover_mean', 
    'wind_speed_10m_mean']
lags_target = [1,2,3,7]
window_rolling_target = [3,7]
lags_dynamic = [1,2]
window_rolling_dynamic = [3]
models = {
    "rain_sum": model_rain,
    "precipitation_hours": model_precip_hours,
    "weather_code": model_weather_code
}

In [131]:
def generate_lag_rolling_features(df_):
    input = df_.groupby('city').tail(1).copy()
    next_date = input['date'] + pd.Timedelta(days=1)
    input['date'] = next_date
    
    for col in target_cols:
        input[col] = np.nan
    
    month_max = 12
    week_day_max = 7
    input['year'] = next_date.dt.year
    input['day_sin'] = np.sin(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)
    input['day_cos'] = np.cos(2 * np.pi * next_date.dt.day / next_date.dt.days_in_month)  
    input['dayofweek_sin'] = np.sin(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['dayofweek_cos'] = np.cos(2 * np.pi * next_date.dt.dayofweek / week_day_max)
    input['month_sin'] = np.sin(2 * np.pi * next_date.dt.month / month_max)
    input['month_cos'] = np.cos(2 * np.pi * next_date.dt.month / month_max)
    input['quarter_sin'] = np.sin(2 * np.pi * next_date.dt.quarter / 4)
    input['quarter_cos'] = np.cos(2 * np.pi * next_date.dt.quarter / 4)
    temp_df = pd.concat([df_, input], ignore_index=True)
    temp_df = temp_df.sort_values(['date', 'city']).reset_index(drop=True)
    for col in target_cols:
        for lag in lags_target:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_target:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
            
    for col in dynamic_cols:
        for lag in lags_dynamic:
            temp_df[f"{col}_lag_{lag}"] = temp_df.groupby('city')[col].shift(lag)
        for window in window_rolling_dynamic:
            temp_df[f"{col}_rolling_mean_{window}"] = temp_df.groupby('city')[col].transform(lambda x: x.rolling(window, min_periods=1).mean())
    final_input = temp_df.groupby('city').tail(1)
    final_input = final_input.drop(columns=side_target + [target_column])
    return final_input, temp_df

In [132]:
def predict_weather(models, data, rolling_window=7): # hiện tại data nhận vào là train_data
    data_sorted = data.sort_values(by=['date', 'city']).reset_index(drop=True)
    predictions = {}
    data_past = data_sorted.groupby('city').tail(rolling_window+7).reset_index(drop=True)
    for i in range(rolling_window):
        input_row, data_past = generate_lag_rolling_features(data_past)
        predictions[f'day_{i+1}'] = {}
        for target, model in models.items():    
            pred = model.predict(input_row)
            predictions[f'day_{i+1}'][target] = pred
            data_past.loc[input_row.index, target] = pred
    return predictions, data_past

In [133]:
class RollingPredictor:
    def __init__(self, models, rolling_window=7):
        self.models = models
        self.rolling_window = rolling_window
        
    def predict(self, data):
        return predict_weather(self.models, data, self.rolling_window)

In [134]:
test_rolling = val_data.groupby('city').head(30).reset_index(drop=True)
test_rolling_predictions, test_rolling_data = predict_weather(models, test_rolling, rolling_window=7)
y_true = val_data.groupby('city').head(37).reset_index(drop=True)
y_true = y_true.groupby('city').tail(7).reset_index(drop=True)

In [135]:
y_true_rain = y_true.groupby('city')['rain_sum'].tail(7).values
y_true_precip_hours = y_true.groupby('city')['precipitation_hours'].tail(7).values
y_true_weather_code = y_true.groupby('city')['weather_code'].tail(7).values

In [136]:
y_pred_rain = []
for target, val in test_rolling_predictions.items():
    y_pred_rain.extend(val['rain_sum'])
y_pred_rain = np.array(y_pred_rain)

In [137]:
y_pred_precip_hours = []
for target, val in test_rolling_predictions.items():
    y_pred_precip_hours.extend(val['precipitation_hours'])
y_pred_precip_hours = np.array(y_pred_precip_hours)

In [138]:
y_pred_weather_code = []
for target, val in test_rolling_predictions.items():
    y_pred_weather_code.extend(val['weather_code'])
y_pred_weather_code = np.array(y_pred_weather_code)

In [139]:
mae = mean_absolute_error(y_true=y_true_rain, y_pred=y_pred_rain)
mse = mean_squared_error(y_true=y_true_rain, y_pred=y_pred_rain)
print(f"Rain Sum - MAE: {mae}, MSE: {mse}")

Rain Sum - MAE: 1.563109448679675, MSE: 5.711567368372143


In [140]:
mae = mean_absolute_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
mse = mean_squared_error(y_true=y_true_precip_hours, y_pred=y_pred_precip_hours)
print(f"Precip Hours - MAE: {mae}, MSE: {mse}")

Precip Hours - MAE: 3.9006826530149246, MSE: 32.135095762124564


In [141]:
rp = classification_report(y_true_weather_code, y_pred_weather_code)
print(rp)

              precision    recall  f1-score   support

           0       0.56      0.76      0.64        38
           1       0.72      0.50      0.59        46

    accuracy                           0.62        84
   macro avg       0.64      0.63      0.62        84
weighted avg       0.65      0.62      0.61        84

